# Run the proverb experiments on Google Colab

Before running:
1. Upload the whole `kannada-maps` folder to **My Drive** (drag and drop it into drive.google.com).
2. In Colab: **Runtime → Change runtime type → T4 GPU**.
3. Run the cells from top to bottom.

Results are written straight into the Drive folder, so nothing is lost if Colab disconnects. Just run the cells again: finished runs are skipped.

In [ ]:
# 1. Connect Google Drive and go to the project folder
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/kannada-maps
!ls

In [ ]:
# 2. Install the same library versions as on the laptop (Colab already has torch)
!pip install -q "transformers==5.5.4" openpyxl accelerate sentencepiece
# Optional speed-up for Qwen3.5's layers; if it fails, everything still works, only slower
!pip install -q flash-linear-attention || echo 'fla not installed, continuing without it'

In [ ]:
# 3. Check the GPU and that all datasets load
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - change the runtime type to T4 GPU!')
!python -m src.data

In [ ]:
# 4. Main run: Qwen3.5-2B on Kannada + English, all three conditions
!python -m src.run_experiment --model qwen3.5-2b --datasets kannada_test kannada_en_reviewed --conditions baseline gold_explanation self_explanation

In [ ]:
# 4b. RERUN after the <think> fix (src/models.py): delete the old Qwen self-explanation
#     results AND the cached explanations, then run only self_explanation again.
!rm -f results/qwen3.5-2b__*__self_explanation.* results/cache/explanations__qwen3.5-2b__*.json
!python -m src.run_experiment --model qwen3.5-2b --datasets kannada_test kannada_en_reviewed --conditions self_explanation
# quick look at the new explanations
import json
for ds in ['kannada_test', 'kannada_en_reviewed']:
    ex = list(json.load(open(f'results/cache/explanations__qwen3.5-2b__{ds}.json', encoding='utf-8')).values())
    print(ds, '| empty:', sum(1 for x in ex if not x.strip()), '| still <think>:', sum('<think>' in x for x in ex))
    for x in ex[:5]: print('   ', x[:120])

In [ ]:
# 5. Summary table (also saved as results/summary.md)
!python -m src.analysis